In [2]:
import pandas as pd
import numpy as np

# Display all columns
pd.set_option("display.max_columns", None)

# Display all rows when needed
pd.set_option("display.max_rows", 100)

In [3]:
customers = pd.read_csv("../data/raw/olist_customers_dataset.csv")

orders = pd.read_csv("../data/raw/olist_orders_dataset.csv")

items = pd.read_csv("../data/raw/olist_order_items_dataset.csv")

payments = pd.read_csv("../data/raw/olist_order_payments_dataset.csv")

reviews = pd.read_csv("../data/raw/olist_order_reviews_dataset.csv")

products = pd.read_csv("../data/raw/olist_products_dataset.csv")

sellers = pd.read_csv("../data/raw/olist_sellers_dataset.csv")

geolocation = pd.read_csv("../data/raw/olist_geolocation_dataset.csv")

translation = pd.read_csv("../data/raw/product_category_name_translation.csv")

In [4]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Order Items": items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers,
    "Geolocation": geolocation,
    "Translation": translation
}

In [5]:
for name, df in datasets.items():
    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Customers: 0 duplicate rows
Orders: 0 duplicate rows
Order Items: 0 duplicate rows
Payments: 0 duplicate rows
Reviews: 0 duplicate rows
Products: 0 duplicate rows
Sellers: 0 duplicate rows
Geolocation: 261831 duplicate rows
Translation: 0 duplicate rows


In [6]:
df = df.drop_duplicates()

In [7]:
missing_summary = pd.DataFrame()

for name, df in datasets.items():
    temp = df.isnull().sum().reset_index()
    temp.columns = ["Column", "Missing Values"]
    temp["Dataset"] = name
    temp = temp[temp["Missing Values"] > 0]

    missing_summary = pd.concat([missing_summary, temp])

missing_summary

,Column,Missing Values,Dataset
4,order_approved_at,160,Orders
5,order_delivered_carrier_date,1783,Orders
6,order_delivered_customer_date,2965,Orders
3,review_comment_title,87656,Reviews
4,review_comment_message,58247,Reviews
1,product_category_name,610,Products
2,product_name_lenght,610,Products
3,product_description_lenght,610,Products
4,product_photos_qty,610,Products
5,product_weight_g,2,Products


convert date column


In [8]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_columns:
    orders[col] = pd.to_datetime(orders[col])

reviews["review_creation_date"] = pd.to_datetime(
    reviews["review_creation_date"]
)

reviews["review_answer_timestamp"] = pd.to_datetime(
    reviews["review_answer_timestamp"]
)

items["shipping_limit_date"] = pd.to_datetime(
    items["shipping_limit_date"]
)

In [9]:
customers = customers.drop_duplicates()
orders = orders.drop_duplicates()
items = items.drop_duplicates()
payments = payments.drop_duplicates()
reviews = reviews.drop_duplicates()
products = products.drop_duplicates()
sellers = sellers.drop_duplicates()
translation = translation.drop_duplicates()

In [10]:
products = products.merge(
    translation,
    on="product_category_name",
    how="left"
)

products.rename(
    columns={
        "product_category_name_english": "category"
    },
    inplace=True
)

products.drop(
    columns=["product_category_name"],
    inplace=True
)

In [11]:
products["category"] = products["category"].fillna("Unknown")

In [12]:
datasets = {
    "Customers": customers,
    "Orders": orders,
    "Items": items,
    "Payments": payments,
    "Reviews": reviews,
    "Products": products,
    "Sellers": sellers
}

summary = []

for name, df in datasets.items():

    summary.append({

        "Dataset": name,

        "Rows": df.shape[0],

        "Columns": df.shape[1],

        "Missing Values": df.isnull().sum().sum()

    })

summary = pd.DataFrame(summary)

summary

,Dataset,Rows,Columns,Missing Values
0,Customers,99441,5,0
1,Orders,99441,8,4908
2,Items,112650,7,0
3,Payments,103886,5,0
4,Reviews,99224,7,145903
5,Products,32951,9,1838
6,Sellers,3095,4,0


In [13]:
products.isnull().sum()

product_id                      0
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
category                        0
dtype: int64

In [14]:
products = products.dropna(
    subset=[
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm"
    ]
)

In [15]:
products.isnull().sum()

product_id                      0
product_name_lenght           609
product_description_lenght    609
product_photos_qty            609
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
category                        0
dtype: int64

In [16]:
customers.to_csv("../data/processed/customers.csv", index=False)

orders.to_csv("../data/processed/orders.csv", index=False)

items.to_csv("../data/processed/items.csv", index=False)

payments.to_csv("../data/processed/payments.csv", index=False)

reviews.to_csv("../data/processed/reviews.csv", index=False)

products.to_csv("../data/processed/products.csv", index=False)

sellers.to_csv("../data/processed/sellers.csv", index=False)